# Cloud input scratch analysis

This notebook is used to inspect the cloud fields that were provided to MYSTIC
and to determine how they should be processed for comparison with the retrieved
shadow displacement.

Goals:

- identify the relevant cloud files
- inspect dimensions, coordinates, and variables
- understand the vertical coordinate
- identify useful cloud metrics such as cloud base, cloud top, cloud fraction,
  mean cloud height, and liquid-water-weighted height
- determine how cloud timestamps correspond to the radiation output

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


In [ ]:
cloud_dir = Path(
    "/work/bb1555/user/kolja/model/libRadtran/data-user/clouds/c3sar-20260606-exp024/v01"
)

files = sorted(cloud_dir.glob("*.nc"))

for f in files:
    print(f.name)

In [ ]:
cloud_files = sorted(cloud_dir.glob("*CLOUD*.nc"))
ic_files = sorted(cloud_dir.glob("*IC*.nc"))
wc_files = sorted(cloud_dir.glob("*WC*.nc"))

print("Cloud files:", len(cloud_files))
print("IC files:", len(ic_files))
print("WC files:", len(wc_files))

In [ ]:
ds_cloud = xr.open_dataset(files[0])

ds_cloud

In [ ]:
cloud_files_dom03 = sorted(
    cloud_dir.glob(
        "CLOUD_exp024_DOM03_res0250m_*.nc"
    )
)

print("Number of DOM03 CLOUD files:", len(cloud_files_dom03))

for f in cloud_files_dom03:
    print(f.name)

In [ ]:
ds_cloud = xr.open_dataset(
    cloud_files_dom03[0]
)

ds_cloud

In [ ]:
print("z:")
print(ds_cloud["z"])
print(ds_cloud["z"].attrs)

print("\nlwc:")
print(ds_cloud["lwc"].attrs)

print("\niwc:")
print(ds_cloud["iwc"].attrs)

In [ ]:
print(ds_cloud.time.values)

In [ ]:
all_cloud_times = []

for f in cloud_files_dom03:
    ds_tmp = xr.open_dataset(f)
    all_cloud_times.extend(ds_tmp.time.values)

all_cloud_times = np.array(all_cloud_times)

print("Total timestamps:", len(all_cloud_times))
print("Unique timestamps:", len(np.unique(all_cloud_times)))

print("\nFirst:")
print(np.sort(np.unique(all_cloud_times))[:5])

print("\nLast:")
print(np.sort(np.unique(all_cloud_times))[-5:])

In [ ]:
cloud_datasets = [
    xr.open_dataset(f)
    for f in cloud_files_dom03
]

ds_cloud_all = xr.concat(
    cloud_datasets,
    dim="time"
)

ds_cloud_all

In [ ]:
print("Dimensions:")
print(ds_cloud_all.sizes)

print("\nFirst time:")
print(ds_cloud_all.time.values[0])

print("\nLast time:")
print(ds_cloud_all.time.values[-1])

print("\nNumber of timestamps:")
print(ds_cloud_all.sizes["time"])

## define Total Water Content
 $ TWC=LWC+IWC $

In [ ]:
ds_cloud_all["twc"] = (
    ds_cloud_all["lwc"]
    + ds_cloud_all["iwc"]
)

ds_cloud_all["twc"].attrs = {
    "long_name": "total cloud water content",
    "units": "kg m-3"
}

In [ ]:
for var in ["lwc", "iwc", "twc"]:
    da = ds_cloud_all[var]

    print(f"\n{var}")
    print("min:", float(da.min()))
    print("max:", float(da.max()))
    print("mean:", float(da.mean()))

In [ ]:
print("z min:", float(ds_cloud_all["z"].min()))
print("z max:", float(ds_cloud_all["z"].max()))
print("first 10 z values:")
print(ds_cloud_all["z"].values[:10])

print("\nlast 10 z values:")
print(ds_cloud_all["z"].values[-10:])

In [ ]:
ds_cloud_all = ds_cloud_all.assign_coords(
    z_m=ds_cloud_all["z"] * 1000
)

ds_cloud_all["z_m"].attrs = {
    "long_name": "geometric height",
    "units": "m",
    "comment": "Converted from z assuming z is given in km"
}

In [ ]:
print("Lowest level:", float(ds_cloud_all["z_m"].min()), "m")
print("Highest level:", float(ds_cloud_all["z_m"].max()), "m")

In [ ]:
thresholds = [
    0.0,
    1e-7,
    1e-6,
    1e-5,
    1e-4
]

for threshold in thresholds:

    cloud_mask = (
        ds_cloud_all["twc"] > threshold
    )

    cloud_fraction_3d = float(
        cloud_mask.mean()
    )

    print(
        f"threshold = {threshold:.1e} kg m-3 "
        f"-> cloudy fraction = {cloud_fraction_3d:.4f}"
    )

In [ ]:
cloud_threshold = 1e-6  # kg m-3

cloud_mask = (
    ds_cloud_all["twc"] > cloud_threshold
)

In [ ]:
z_m = ds_cloud_all["z_m"]

# Height field broadcast onto the cloud mask
cloud_height = z_m.where(cloud_mask)

# Cloud base and top for every horizontal column
cloud_base = cloud_height.min(dim="z")
cloud_top = cloud_height.max(dim="z")

# Whether a horizontal column contains any cloud
cloudy_column = cloud_mask.any(dim="z")

In [ ]:
cloud_center = (
    cloud_base + cloud_top
) / 2

cloud_thickness = (
    cloud_top - cloud_base
)

In [ ]:
cloud_fraction = (
    cloudy_column.mean(
        dim=("lat", "lon")
    )
)

In [ ]:
ti = 16

cb = cloud_base.isel(time=ti)
ct = cloud_top.isel(time=ti)
cc = cloud_center.isel(time=ti)
th = cloud_thickness.isel(time=ti)

print("Time:", ds_cloud_all.time.values[ti])

print("\nCloud base [m]")
print("mean:", float(cb.mean(skipna=True)))
print("median:", float(cb.median(skipna=True)))
print("min:", float(cb.min(skipna=True)))
print("max:", float(cb.max(skipna=True)))

print("\nCloud top [m]")
print("mean:", float(ct.mean(skipna=True)))
print("median:", float(ct.median(skipna=True)))
print("min:", float(ct.min(skipna=True)))
print("max:", float(ct.max(skipna=True)))

print("\nCloud center [m]")
print("mean:", float(cc.mean(skipna=True)))
print("median:", float(cc.median(skipna=True)))

print("\nCloud thickness [m]")
print("mean:", float(th.mean(skipna=True)))
print("median:", float(th.median(skipna=True)))

print(
    "\nCloud fraction:",
    float(cloud_fraction.isel(time=ti))
)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(13, 4),
    constrained_layout=True
)

axes[0].hist(
    cb.values[np.isfinite(cb.values)],
    bins=30
)
axes[0].set_xlabel("Cloud base height [m]")
axes[0].set_ylabel("Number of columns")
axes[0].set_title("Cloud base")

axes[1].hist(
    ct.values[np.isfinite(ct.values)],
    bins=30
)
axes[1].set_xlabel("Cloud top height [m]")
axes[1].set_title("Cloud top")

axes[2].hist(
    th.values[np.isfinite(th.values)],
    bins=30
)
axes[2].set_xlabel("Cloud thickness [m]")
axes[2].set_title("Cloud thickness")

for ax in axes:
    ax.grid()

plt.show()

### TWC-weighted height for each horizontal column

In [ ]:
twc = ds_cloud_all["twc"]
z_m = ds_cloud_all["z_m"]

twc_sum = twc.sum(dim="z")

twc_weighted_height = (
    (twc * z_m).sum(dim="z")
    / twc_sum
)

# Cloud-free columns -> NaN
twc_weighted_height = twc_weighted_height.where(
    twc_sum > 0
)

In [ ]:
ti = 16

z_twc = twc_weighted_height.isel(time=ti)

print("Time:", ds_cloud_all.time.values[ti])

print("\nTWC-weighted height [m]")
print("mean:", float(z_twc.mean(skipna=True)))
print("median:", float(z_twc.median(skipna=True)))
print("min:", float(z_twc.min(skipna=True)))
print("max:", float(z_twc.max(skipna=True)))

In [ ]:
plt.figure(figsize=(7, 5))

plt.hist(
    z_twc.values[np.isfinite(z_twc.values)],
    bins=30
)

plt.xlabel("TWC-weighted height [m]")
plt.ylabel("Number of columns")
plt.title("TWC-weighted cloud height")
plt.grid()

plt.show()

In [ ]:
twc_cloud = ds_cloud_all["twc"].where(
    cloud_mask
)

twc_sum = twc_cloud.sum(dim="z")

twc_weighted_height = (
    (twc_cloud * z_m).sum(dim="z")
    / twc_sum
)

twc_weighted_height = twc_weighted_height.where(
    twc_sum > 0
)

## Cloud-height metrics

The aim of this analysis is to derive representative cloud-height metrics that
can later be compared with the effective height inferred from the displacement
of the 1D and 3D cloud-shadow fields,

$$
h_{\mathrm{eff}} = \frac{d}{\tan(\mathrm{SZA})}.
$$

It is not clear a priori which physical cloud height should correspond most
closely to this effective displacement height. The shadow pattern results from
the three-dimensional cloud field rather than from a single vertical level.

Therefore, several candidate cloud-height metrics are considered instead of
assuming a single definition:

- cloud-base height
- cloud-top height
- geometric cloud-center height
- cloud thickness
- total-water-content-weighted cloud height

The relationship between these quantities and the displacement-derived
effective height will be investigated later.

### Cloud condensate and cloud definition

Liquid water content (`lwc`) and ice water content (`iwc`) are combined into
total cloud water content,

$$
\mathrm{TWC} = \mathrm{LWC} + \mathrm{IWC}.
$$

TWC is used here as a convenient combined condensate field for identifying
cloud geometry. It is not interpreted as a standalone radiative quantity.

A grid cell is classified as cloudy when

$$
\mathrm{TWC} > 10^{-6}\ \mathrm{kg\,m^{-3}}.
$$

A sensitivity test showed that the fraction of cloudy 3-D grid cells is
essentially unchanged between thresholds of 0 and $10^{-6}\,\mathrm{kg\,m^{-3}}$,
while larger thresholds begin to remove a substantial fraction of the cloud
field.

The original vertical coordinate `z` has no unit attribute in the source file.
Based on its range (approximately 0.144–21.3), it is interpreted as geometric
height in kilometres and converted to metres for the following analysis.

### Column-wise cloud geometry

Cloud geometry is first determined independently for each horizontal
`(lat, lon)` column.

For every cloudy column:

- **cloud base** is the lowest cloudy model level,
- **cloud top** is the highest cloudy model level,
- **cloud center** is defined as the midpoint between cloud base and cloud top,
- **cloud thickness** is the difference between cloud top and cloud base.

Cloud-free columns are excluded from the height statistics.

The resulting column-wise distributions are summarized over the domain using
both the mean and median. The median provides a robust estimate of a typical
cloud column, while the mean retains sensitivity to taller or deeper clouds.

In [ ]:
ti = 16  # 12:00 UTC

cb = cloud_base.isel(time=ti)
ct = cloud_top.isel(time=ti)
cc = cloud_center.isel(time=ti)
th = cloud_thickness.isel(time=ti)

fig, axes = plt.subplots(
    2, 2,
    figsize=(10, 8),
    constrained_layout=True
)

plot_data = [
    (cb, "Cloud base", "Cloud-base height [m]"),
    (ct, "Cloud top", "Cloud-top height [m]"),
    (cc, "Geometric cloud center", "Cloud-center height [m]"),
    (th, "Cloud thickness", "Cloud thickness [m]")
]

for ax, (data, title, xlabel) in zip(
    axes.flat,
    plot_data
):
    values = data.values
    values = values[np.isfinite(values)]

    ax.hist(
        values,
        bins=30
    )

    ax.axvline(
        np.mean(values),
        linestyle="--",
        label="Mean"
    )

    ax.axvline(
        np.median(values),
        linestyle=":",
        label="Median"
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of cloudy columns")
    ax.grid()
    ax.legend()

fig.suptitle(
    f"Cloud geometry at {pd.to_datetime(ds_cloud_all.time.values[ti]):%Y-%m-%d %H:%M}"
)

plt.show()

### TWC-weighted cloud height

In addition to purely geometric cloud-height measures, a condensate-weighted
height is calculated for every cloudy column:

$$
z_{\mathrm{TWC}}
=
\frac{\sum_z z\,\mathrm{TWC}}
     {\sum_z \mathrm{TWC}}.
$$

Only grid cells satisfying the cloud-mask criterion are included.

This quantity represents the vertical center of the cloud condensate rather
than simply the geometric midpoint between cloud base and cloud top. It is
included as an additional candidate for comparison with the
displacement-derived effective height.

In [ ]:
twc_cloud = ds_cloud_all["twc"].where(
    cloud_mask
)

twc_sum = twc_cloud.sum(dim="z")

twc_weighted_height = (
    (twc_cloud * z_m).sum(dim="z")
    / twc_sum
)

twc_weighted_height = twc_weighted_height.where(
    twc_sum > 0
)

In [ ]:
z_twc = twc_weighted_height.isel(time=ti)

values = z_twc.values
values = values[np.isfinite(values)]

fig, ax = plt.subplots(
    figsize=(7, 5),
    constrained_layout=True
)

ax.hist(
    values,
    bins=30
)

ax.axvline(
    np.mean(values),
    linestyle="--",
    label=f"Mean = {np.mean(values):.0f} m"
)

ax.axvline(
    np.median(values),
    linestyle=":",
    label=f"Median = {np.median(values):.0f} m"
)

ax.set_xlabel(
    "TWC-weighted cloud height [m]"
)

ax.set_ylabel(
    "Number of cloudy columns"
)

ax.set_title(
    "TWC-weighted cloud-height distribution"
)

ax.grid()
ax.legend()

plt.show()

In [ ]:
labels = [
    "Cloud base",
    "Cloud center",
    "TWC-weighted",
    "Cloud top"
]

means = [
    float(cb.mean(skipna=True)),
    float(cc.mean(skipna=True)),
    float(z_twc.mean(skipna=True)),
    float(ct.mean(skipna=True))
]

medians = [
    float(cb.median(skipna=True)),
    float(cc.median(skipna=True)),
    float(z_twc.median(skipna=True)),
    float(ct.median(skipna=True))
]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(
    figsize=(8, 5),
    constrained_layout=True
)

ax.bar(
    x - width / 2,
    means,
    width,
    label="Mean"
)

ax.bar(
    x + width / 2,
    medians,
    width,
    label="Median"
)

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_ylabel(
    "Height [m]"
)

ax.set_title(
    f"Representative cloud heights at "
    f"{pd.to_datetime(ds_cloud_all.time.values[ti]):%H:%M}"
)

ax.grid(
    axis="y"
)

ax.legend()

plt.show()

In [ ]:
cb_median = cloud_base.median(
    dim=("lat", "lon"),
    skipna=True
)

ct_median = cloud_top.median(
    dim=("lat", "lon"),
    skipna=True
)

cc_median = cloud_center.median(
    dim=("lat", "lon"),
    skipna=True
)

z_twc_median = twc_weighted_height.median(
    dim=("lat", "lon"),
    skipna=True
)

fig, ax = plt.subplots(
    figsize=(10, 5),
    constrained_layout=True
)

ax.plot(
    ds_cloud_all.time,
    cb_median,
    marker="o",
    label="Cloud base"
)

ax.plot(
    ds_cloud_all.time,
    cc_median,
    marker="o",
    label="Geometric center"
)

ax.plot(
    ds_cloud_all.time,
    z_twc_median,
    marker="o",
    label="TWC-weighted height"
)

ax.plot(
    ds_cloud_all.time,
    ct_median,
    marker="o",
    label="Cloud top"
)

ax.set_xlabel("Time")
ax.set_ylabel("Median height [m]")

ax.set_title(
    "Evolution of representative cloud heights"
)

ax.grid()
ax.legend()

plt.show()

In [ ]:
thickness_median = cloud_thickness.median(
    dim=("lat", "lon"),
    skipna=True
)

fig, ax1 = plt.subplots(
    figsize=(10, 5),
    constrained_layout=True
)

ax1.plot(
    ds_cloud_all.time,
    cloud_fraction,
    marker="o",
    label="Cloud fraction"
)

ax1.set_xlabel("Time")
ax1.set_ylabel("Cloud fraction")
ax1.grid()

ax2 = ax1.twinx()

ax2.plot(
    ds_cloud_all.time,
    thickness_median,
    marker="o",
    linestyle="--",
    label="Median cloud thickness"
)

ax2.set_ylabel(
    "Median cloud thickness [m]"
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best"
)

ax1.set_title(
    "Cloud fraction and cloud thickness"
)

plt.show()

## Candidate metrics for further analysis

The cloud field shows substantial spatial variability in cloud base, cloud top,
and cloud thickness. Therefore, no single cloud-height definition is selected
a priori.

For each timestamp, the final cloud-processing workflow will retain:

- horizontal cloud fraction,
- mean and median cloud-base height,
- mean and median cloud-top height,
- mean and median geometric cloud-center height,
- mean and median cloud thickness,
- mean and median TWC-weighted cloud height.

These quantities will be exported as a timestamp-based table and compared with
the effective shadow-displacement height derived independently from the
radiation calculations.

The comparison will then determine which cloud-height metric best describes
the height scale represented by the retrieved shadow displacement.